# MindSync Stage 2 v3 — wav2vec2-Large-960h audio classifier, corrected pooling, 3 seeds

**Same checkpoint as v1** (`facebook/wav2vec2-large-960h`). v1 (test acc 0.4056 / macro-F1 0.3996) mean-pooled the
**last** hidden layer over zero-padded frames. v3 uses `AudioModelV3` (pasted verbatim below):
1. learned softmax-weighted sum over all 25 hidden states (each LayerNorm-ed first);
2. mean + std pooling over **valid frames only** (frame mask from the true sample lengths);
3. train-only waveform augmentation (speed 0.9–1.1, gain 0.8–1.2, noise 15–30 dB SNR with p = 0.5) + light SpecAugment.

**Unchanged from v1, so the test number is comparable:** RAVDESS speech (1,440 clips); emotion → cluster mapping
(calm → Resilience); speaker-independent split actors 1–18 / 19–21 / 22–24; librosa 16 kHz mono, `trim(top_db=30)`,
5 s centre crop for val/test.

**Selection uses validation macro-F1 only.** Test is evaluated once per seed on that seed's best-val checkpoint.
Seeds 42, 43, 44 → mean ± std.

**Outputs (`/kaggle/working`)**
- `stage2_v3_results.json` — rewritten after every seed (config, per-seed metrics, reports, confusion matrices, layer weights, history)
- `audio_v3_best.pt` — the single checkpoint kept on disk: the seed with the highest **val** macro-F1
- the last cell prints the block between `===== SUMMARY START =====` and `===== SUMMARY END =====`

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.45.2', 'datasets==2.21.0', 'huggingface_hub==0.25.2'], check=True)

import os, gc, json, math, time, random, pathlib, urllib.request, zipfile, collections, traceback, warnings
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings('ignore', category=FutureWarning)

T_START = time.time()
WORK = pathlib.Path('/kaggle/working'); WORK.mkdir(parents=True, exist_ok=True)
DATA_DIR = pathlib.Path('/tmp/ravdess'); DATA_DIR.mkdir(parents=True, exist_ok=True)   # outside /kaggle/working -> not saved as output
RESULTS_PATH = WORK / 'stage2_v3_results.json'
BEST_CKPT = WORK / 'audio_v3_best.pt'

assert torch.cuda.is_available(), 'No GPU - enable the T4 accelerator'
DEVICE = torch.device('cuda')
AMP = DEVICE.type == 'cuda'

import transformers, librosa, sklearn
transformers.logging.set_verbosity_error()
print('python', sys.version.split()[0], '| torch', torch.__version__, '| transformers', transformers.__version__,
      '| librosa', librosa.__version__, '| sklearn', sklearn.__version__,
      '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu', flush=True)

## 1 · Model class — pasted VERBATIM from `notebooks_v3/audio_model_v3.py`
Do not edit: the Stage 3 v3 notebook loads the saved `state_dict` into this exact class.
For `-960h` (`feat_extract_norm == 'group'`) the encoder gets **no** `attention_mask`; inputs are plain
zero-padded and the frame mask comes from the true `lengths`.

In [ ]:
"""Shared Stage 2 v3 audio model (wav2vec2-Large-960h, same checkpoint as v1).

Fixes relative to v1 (which mean-pooled the LAST layer over zero-padding):
  1. learned softmax-weighted sum over all 25 hidden states (each LayerNorm-ed first)
  2. mean+std pooling over VALID frames only (frame mask from true input lengths)
  3. (training side) waveform augmentation + light SpecAugment

Both the Stage 2 v3 training notebook and the Stage 3 v3 notebook paste this
class VERBATIM so the saved state_dict always loads.
"""
import torch
import torch.nn as nn
from transformers import Wav2Vec2Config, Wav2Vec2Model

AUDIO_MODEL_NAME = 'facebook/wav2vec2-large-960h'


class AudioModelV3(nn.Module):
    def __init__(self, model_name=AUDIO_MODEL_NAME, num_classes=4, freeze_layers=8,
                 dropout=0.1, spec_augment=True):
        super().__init__()
        cfg = Wav2Vec2Config.from_pretrained(model_name)
        cfg.output_hidden_states = True
        cfg.apply_spec_augment = spec_augment
        cfg.mask_time_prob = 0.05
        cfg.mask_time_length = 5
        cfg.mask_feature_prob = 0.0
        self.enc = Wav2Vec2Model.from_pretrained(model_name, config=cfg)
        self.enc.freeze_feature_encoder()
        for layer in self.enc.encoder.layers[:freeze_layers]:
            for p in layer.parameters():
                p.requires_grad = False
        # 'layer' norm models accept attention_mask; 'group' norm models (e.g. -960h) must not get one
        self.use_attention_mask = (cfg.feat_extract_norm == 'layer')
        H = cfg.hidden_size
        L = cfg.num_hidden_layers + 1
        self.layer_w = nn.Parameter(torch.zeros(L))
        self.layer_norm = nn.LayerNorm(H)
        self.proj = nn.Sequential(nn.LayerNorm(2 * H), nn.Linear(2 * H, H))
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(H, H // 2), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(H // 2, num_classes))

    def frames(self, input_values, lengths):
        """input_values: (B, S) zero-padded waveforms; lengths: (B,) true sample counts.
        Returns frame features (B, T, H) and a bool frame mask (B, T), True = valid."""
        if self.use_attention_mask:
            am = (torch.arange(input_values.shape[1], device=input_values.device)[None, :]
                  < lengths[:, None]).long()
            out = self.enc(input_values=input_values, attention_mask=am, output_hidden_states=True)
        else:
            out = self.enc(input_values=input_values, output_hidden_states=True)
        hs = torch.stack([self.layer_norm(h) for h in out.hidden_states], 0)   # (L, B, T, H)
        w = torch.softmax(self.layer_w, 0)
        f = (w[:, None, None, None] * hs).sum(0)                               # (B, T, H)
        flen = self.enc._get_feat_extract_output_lengths(lengths).clamp(min=1)
        mask = torch.arange(f.shape[1], device=f.device)[None, :] < flen[:, None]
        return f, mask

    @staticmethod
    def pool(f, mask):
        m = mask.unsqueeze(-1).to(f.dtype)
        n = m.sum(1).clamp_min(1.0)
        mean = (f * m).sum(1) / n
        var = (((f - mean[:, None, :]) ** 2) * m).sum(1) / n
        return torch.cat([mean, var.clamp_min(1e-6).sqrt()], -1)

    def forward(self, input_values, lengths):
        f, mask = self.frames(input_values, lengths)
        emb = self.proj(self.pool(f, mask))
        return {'embedding': emb, 'logits': self.head(emb)}

## 2 · RAVDESS speech (Zenodo, same zip as v1)

In [ ]:
ZIP = DATA_DIR / 'ravdess.zip'
AUDIO_ROOT = DATA_DIR / 'extracted'
if not AUDIO_ROOT.exists() or len(list(AUDIO_ROOT.rglob('*.wav'))) < 1440:
    if not ZIP.exists() or ZIP.stat().st_size < 100_000_000:
        print('Downloading RAVDESS speech (~200 MB) from Zenodo ...', flush=True)
        urllib.request.urlretrieve('https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip', ZIP)
    AUDIO_ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(AUDIO_ROOT)
    ZIP.unlink()
wavs = sorted(AUDIO_ROOT.rglob('*.wav'))
print('RAVDESS wav files:', len(wavs), flush=True)

## 3 · Labels + speaker-independent split (identical to v1)
Filename `MM-VV-EE-II-SS-RR-AA.wav`: field 3 = emotion, field 7 = actor.

In [ ]:
CLUSTER_NAMES = ['Distress', 'Resilience', 'Aggression', 'Ambiguity']
C2I = {c: i for i, c in enumerate(CLUSTER_NAMES)}
RAVDESS_TO_CLUSTER = {1: 'Ambiguity',    # neutral
                      2: 'Resilience',   # calm  (v1 mapping - NOT the calm->Ambiguity of the v2 notebook)
                      3: 'Resilience',   # happy
                      4: 'Distress',     # sad
                      5: 'Aggression',   # angry
                      6: 'Distress',     # fearful
                      7: 'Aggression',   # disgust
                      8: 'Ambiguity'}    # surprised

by_name = {}
for p in wavs:
    if len(p.stem.split('-')) == 7:
        by_name.setdefault(p.name, p)          # de-duplicate by filename, just in case
items = []                                     # (path, cluster_idx, actor)
for name, p in sorted(by_name.items()):
    parts = p.stem.split('-')
    emo, actor = int(parts[2]), int(parts[6])
    items.append((str(p), C2I[RAVDESS_TO_CLUSTER[emo]], actor))
assert len(items) == 1440, f'expected 1440 RAVDESS speech clips, got {len(items)}'

SPLITS = {'train': [s for s in items if s[2] <= 18],
          'val':   [s for s in items if 19 <= s[2] <= 21],
          'test':  [s for s in items if s[2] >= 22]}
ACTORS = {k: {s[2] for s in v} for k, v in SPLITS.items()}
assert ACTORS['train'] == set(range(1, 19)) and ACTORS['val'] == {19, 20, 21} and ACTORS['test'] == {22, 23, 24}
assert not (ACTORS['train'] & ACTORS['val']) and not (ACTORS['train'] & ACTORS['test']) and not (ACTORS['val'] & ACTORS['test'])
assert [len(SPLITS[k]) for k in ('train', 'val', 'test')] == [1080, 180, 180]
for k, v in SPLITS.items():
    cnt = collections.Counter(s[1] for s in v)
    print(f'{k:5s}: {len(v):4d} clips | actors {min(ACTORS[k])}-{max(ACTORS[k])} | '
          + ', '.join(f'{CLUSTER_NAMES[c]} {cnt[c]}' for c in range(4)))
print('speaker-disjoint: OK')

## 4 · Preprocessing, augmentation, collate
- decode once: librosa 16 kHz mono + `trim(top_db=30)` (as v1), kept in RAM
- train: augmentation, then a **random** 5 s crop · val/test: **centre** 5 s crop (as v1)
- `Wav2Vec2FeatureExtractor` (-960h) zero-mean / unit-variance normalisation is applied to **each clip on its own,
  before padding** (calling it on a padded batch without an attention mask would include the padding in the statistics)
- collate: zero-padded `input_values`, `lengths` = true sample counts, `labels`

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import Wav2Vec2FeatureExtractor

SR, MAX_SAMPLES, MIN_SAMPLES = 16_000, 80_000, 400      # 5 s crop; 400 samples = wav2vec2 receptive field
fe = Wav2Vec2FeatureExtractor.from_pretrained(AUDIO_MODEL_NAME)
print(f'feature extractor: do_normalize={fe.do_normalize} | padding_value={fe.padding_value} | '
      f'sampling_rate={fe.sampling_rate} | return_attention_mask={fe.return_attention_mask}')
assert fe.do_normalize and fe.sampling_rate == SR

t0 = time.time(); WAV = {}
for i, (path, _, _) in enumerate(items):
    w, _ = librosa.load(path, sr=SR, mono=True)
    w, _ = librosa.effects.trim(w, top_db=30)
    WAV[path] = w.astype(np.float32)
    if (i + 1) % 360 == 0:
        print(f'  decoded {i + 1}/{len(items)}', flush=True)
secs = np.array([len(w) for w in WAV.values()]) / SR
print(f'decoded in {time.time() - t0:.0f}s | trimmed length mean {secs.mean():.2f}s, min {secs.min():.2f}s, '
      f'max {secs.max():.2f}s | longer than 5 s: {(secs > 5).mean():.1%}', flush=True)


def crop(w, train):
    if len(w) <= MAX_SAMPLES:
        return w
    s = random.randint(0, len(w) - MAX_SAMPLES) if train else (len(w) - MAX_SAMPLES) // 2   # eval: CENTRE crop
    return w[s:s + MAX_SAMPLES]


def augment(w):
    speed = random.uniform(0.9, 1.1)                          # speed perturbation via resampling
    w = librosa.resample(w, orig_sr=SR, target_sr=int(round(SR / speed)))
    w = w * random.uniform(0.8, 1.2)                          # gain (kept as specified; the per-clip normalisation
                                                              #       below makes it almost a no-op)
    if random.random() < 0.5:                                 # additive white noise at 15-30 dB SNR
        snr_db = random.uniform(15.0, 30.0)
        rms = float(np.sqrt(np.mean(w ** 2))) + 1e-9
        w = w + np.random.normal(0.0, rms / 10 ** (snr_db / 20), len(w))
    return w.astype(np.float32)


class RavdessDS(Dataset):
    """(normalised waveform, label). Train: augment + random 5 s crop. Eval: centre 5 s crop."""
    def __init__(self, rows, train):
        self.rows, self.train = rows, train

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        path, y, _ = self.rows[i]
        w = WAV[path]
        if self.train:
            w = augment(w)      # python `random` / numpy are re-seeded per worker per epoch from the loader's generator
        w = crop(w, self.train)
        if len(w) < MIN_SAMPLES:
            w = np.pad(w, (0, MIN_SAMPLES - len(w)))
        x = fe(w, sampling_rate=SR, return_tensors='np')['input_values'][0].astype(np.float32)   # this clip only
        return x, y


def collate(batch):
    """Zero-pads to the longest clip. `lengths` = true sample counts (after crop + normalisation)."""
    lengths = torch.tensor([len(x) for x, _ in batch], dtype=torch.long)
    iv = torch.zeros(len(batch), int(lengths.max()), dtype=torch.float32)
    for i, (x, _) in enumerate(batch):
        iv[i, :len(x)] = torch.from_numpy(x)
    return {'input_values': iv, 'lengths': lengths,
            'labels': torch.tensor([y for _, y in batch], dtype=torch.long)}


_ds = RavdessDS(SPLITS['val'], False)
_b = collate([_ds[0], _ds[7]])
_L = _b['lengths'].tolist()
assert _b['input_values'].shape == (2, max(_L)) and max(_L) <= MAX_SAMPLES
for _i, _l in enumerate(_L):
    assert torch.all(_b['input_values'][_i, _l:] == 0)
print('collate check: input_values', tuple(_b['input_values'].shape), '| lengths', _L, '| labels', _b['labels'].tolist())

## 5 · Training configuration

In [ ]:
SEEDS          = [42, 43, 44]
FREEZE_LAYERS  = 8            # AudioModelV3 default; stored in the checkpoint
BATCH, ACCUM   = 1, 16        # effective batch 16; batch 1 => no zero-padding ever reaches the -960h encoder
                              # (GroupNorm frontend, no attention_mask), so training matches eval and Stage 3 extraction
EVAL_BATCH     = 1            # val/test one clip at a time -> no padding reaches the -960h encoder (it gets no attention_mask)
NUM_WORKERS    = 2
LR_BACKBONE    = 1e-5         # unfrozen wav2vec2 parameters
LR_NEW         = 3e-4         # layer_w, layer_norm, proj, head
WEIGHT_DECAY   = 0.01
WARMUP_FRAC    = 0.10
LABEL_SMOOTH   = 0.05
GRAD_CLIP      = 1.0
MAX_EPOCHS     = 20
PATIENCE       = 5            # early stopping on val macro-F1
TIME_BUDGET_H  = 11.0         # Kaggle hard limit is 12 h: no new epoch / seed is started after this
V1_TEST        = {'acc': 0.4056, 'f1': 0.3996}

CONFIG = dict(model_name=AUDIO_MODEL_NAME, model_class='AudioModelV3', freeze_layers=FREEZE_LAYERS, seeds=SEEDS,
              batch=BATCH, grad_accum=ACCUM, eval_batch=EVAL_BATCH, lr_backbone=LR_BACKBONE, lr_new=LR_NEW,
              weight_decay=WEIGHT_DECAY, warmup_frac=WARMUP_FRAC, label_smoothing=LABEL_SMOOTH, grad_clip=GRAD_CLIP,
              max_epochs=MAX_EPOCHS, patience=PATIENCE, amp=AMP, sampler='class-balanced WeightedRandomSampler',
              augmentation='speed U(0.9,1.1) via resampling, gain U(0.8,1.2), white noise SNR U(15,30) dB p=0.5; SpecAugment in encoder',
              preprocessing='librosa 16 kHz mono, trim top_db=30, 5 s crop (train random / eval centre), per-clip Wav2Vec2FeatureExtractor normalisation',
              split='speaker-independent: train actors 1-18 / val 19-21 / test 22-24',
              cluster_map={str(k): v for k, v in RAVDESS_TO_CLUSTER.items()}, cluster_names=CLUSTER_NAMES,
              selection='best epoch and saved seed chosen by VALIDATION macro-F1', v1_test=V1_TEST,
              versions={'torch': torch.__version__, 'transformers': transformers.__version__, 'librosa': librosa.__version__})
print(json.dumps(CONFIG, indent=1))

## 6 · Train / evaluate helpers
Two-group AdamW, cosine schedule with 10% warm-up, AMP, grad-accumulation, label smoothing, grad clipping,
early stopping on val macro-F1, initial-loss sanity check (≈ ln 4 = 1.386; prints PASS/FAIL, never aborts).
The best epoch's weights are kept in CPU RAM, not on disk.

In [ ]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report, confusion_matrix

LABELS = [0, 1, 2, 3]


def hours():
    return (time.time() - T_START) / 3600


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP)


def _json_default(o):
    return o.item() if hasattr(o, 'item') else str(o)


def make_loaders(seed):
    rows = SPLITS['train']
    counts = np.bincount([r[1] for r in rows], minlength=4)
    weights = torch.tensor([1.0 / counts[r[1]] for r in rows], dtype=torch.double)
    sampler = WeightedRandomSampler(weights, num_samples=len(rows), replacement=True,
                                    generator=torch.Generator().manual_seed(seed))
    kw = dict(num_workers=NUM_WORKERS, pin_memory=AMP, collate_fn=collate,
              multiprocessing_context='fork' if NUM_WORKERS > 0 else None)
    train = DataLoader(RavdessDS(rows, True), batch_size=BATCH, sampler=sampler,
                       generator=torch.Generator().manual_seed(seed + 10_000), **kw)
    val = DataLoader(RavdessDS(SPLITS['val'], False), batch_size=EVAL_BATCH, shuffle=False, **kw)
    test = DataLoader(RavdessDS(SPLITS['test'], False), batch_size=EVAL_BATCH, shuffle=False, **kw)
    return train, val, test


def forward_logits(model, b):
    return model(b['input_values'].to(DEVICE, non_blocking=True),
                 b['lengths'].to(DEVICE, non_blocking=True))['logits']


@torch.no_grad()
def evaluate(model, loader):
    model.eval(); P, Y = [], []
    for b in loader:
        with autocast():
            logits = forward_logits(model, b)
        P += logits.float().argmax(-1).cpu().tolist(); Y += b['labels'].tolist()
    P, Y = np.array(P), np.array(Y)
    return {'acc': float((P == Y).mean()),
            'f1': float(f1_score(Y, P, labels=LABELS, average='macro', zero_division=0)),
            'report': classification_report(Y, P, labels=LABELS, target_names=CLUSTER_NAMES, digits=4,
                                            output_dict=True, zero_division=0),
            'confusion': confusion_matrix(Y, P, labels=LABELS).tolist(),
            'n': int(len(Y))}


SANITY_ROWS = random.Random(0).sample(SPLITS['val'], 32)     # fixed val subset, same for every seed


@torch.no_grad()
def initial_loss_check(model):
    loader = DataLoader(RavdessDS(SANITY_ROWS, False), batch_size=BATCH, shuffle=False, collate_fn=collate)
    model.eval(); tot, n, stds = 0.0, 0, []
    for b in loader:
        with autocast():
            logits = forward_logits(model, b)
        tot += F.cross_entropy(logits.float(), b['labels'].to(DEVICE), reduction='sum').item()
        n += len(b['labels']); stds.append(logits.float().std().item())
    loss = tot / n
    status = 'PASS' if abs(loss - math.log(4)) < 0.3 else 'FAIL'
    print(f'  initial-loss sanity check: {loss:.4f} (expected ~ ln 4 = {math.log(4):.4f}; '
          f'logit std {np.mean(stds):.3f}) -> {status}', flush=True)
    return float(loss), status


def run_seed(seed):
    set_seed(seed)
    model = AudioModelV3(model_name=AUDIO_MODEL_NAME, num_classes=4, freeze_layers=FREEZE_LAYERS).to(DEVICE)
    new_params = [model.layer_w, *model.layer_norm.parameters(), *model.proj.parameters(), *model.head.parameters()]
    backbone_params = [p for p in model.enc.parameters() if p.requires_grad]
    trainable = [p for p in model.parameters() if p.requires_grad]
    assert not ({id(p) for p in new_params} & {id(p) for p in backbone_params})
    assert len(new_params) + len(backbone_params) == len(trainable)
    print(f'  params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M total | '
          f'backbone trainable {sum(p.numel() for p in backbone_params) / 1e6:.1f} M (lr {LR_BACKBONE}) | '
          f'new {sum(p.numel() for p in new_params) / 1e6:.2f} M (lr {LR_NEW})', flush=True)

    train_loader, val_loader, test_loader = make_loaders(seed)
    init_loss, init_status = initial_loss_check(model)

    optim = AdamW([{'params': backbone_params, 'lr': LR_BACKBONE},
                   {'params': new_params, 'lr': LR_NEW}], weight_decay=WEIGHT_DECAY)
    steps_per_epoch = math.ceil(len(train_loader) / ACCUM)
    total_steps = steps_per_epoch * MAX_EPOCHS
    sched = get_cosine_schedule_with_warmup(optim, int(WARMUP_FRAC * total_steps), total_steps)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=AMP)
    crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
    print(f'  {len(train_loader)} batches x {BATCH} -> {steps_per_epoch} optimiser steps/epoch, '
          f'{total_steps} max steps, warm-up {int(WARMUP_FRAC * total_steps)}', flush=True)
    if AMP:
        torch.cuda.reset_peak_memory_stats()

    best_f1, best_epoch, bad, best_state, history, stop_reason = -1.0, 0, 0, None, [], 'max_epochs'
    t_train = time.time()
    for ep in range(1, MAX_EPOCHS + 1):
        if hours() > TIME_BUDGET_H:
            stop_reason = 'time_budget'
            print(f'  time budget ({TIME_BUDGET_H} h) reached - stopping this seed', flush=True)
            break
        model.train(); t0 = time.time()
        loss_sum, correct, seen, nonfinite = 0.0, 0, 0, 0
        optim.zero_grad(set_to_none=True)
        for i, b in enumerate(train_loader):
            y = b['labels'].to(DEVICE, non_blocking=True)
            with autocast():
                logits = forward_logits(model, b)
            loss = crit(logits.float(), y)
            scaler.scale(loss / ACCUM).backward()
            if (i + 1) % ACCUM == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
                scaler.step(optim); scaler.update()
                optim.zero_grad(set_to_none=True)
                sched.step()
            if torch.isfinite(loss):
                loss_sum += loss.item() * len(y)
            else:
                nonfinite += 1
            seen += len(y); correct += int((logits.argmax(-1) == y).sum())
            if (i + 1) % 90 == 0:
                print(f'    ep {ep} step {i + 1}/{len(train_loader)} | running loss {loss_sum / max(seen, 1):.4f}', flush=True)

        v = evaluate(model, val_loader)
        lrs = sched.get_last_lr()
        history.append({'epoch': ep, 'train_loss': loss_sum / max(seen, 1), 'train_acc': correct / max(seen, 1),
                        'val_acc': v['acc'], 'val_f1': v['f1'], 'lr_backbone': lrs[0], 'lr_new': lrs[1],
                        'nonfinite_losses': nonfinite, 'epoch_sec': time.time() - t0})
        mark = ''
        if v['f1'] > best_f1:
            best_f1, best_epoch, bad = v['f1'], ep, 0
            best_state = {k: t.detach().to('cpu', copy=True) for k, t in model.state_dict().items()}
            mark = '  * best'
        else:
            bad += 1
        print(f'  seed {seed} | epoch {ep:2d} | train loss {history[-1]["train_loss"]:.4f} acc {history[-1]["train_acc"]:.4f} | '
              f'val acc {v["acc"]:.4f} macro-F1 {v["f1"]:.4f} | {time.time() - t0:.0f}s | elapsed {hours():.2f} h{mark}', flush=True)
        if bad >= PATIENCE:
            stop_reason = 'early_stopping'
            print(f'  early stop: {PATIENCE} epochs without val macro-F1 improvement', flush=True)
            break

    if best_state is None:
        raise RuntimeError('no epoch completed (time budget)')
    model.load_state_dict(best_state)
    val_m, test_m = evaluate(model, val_loader), evaluate(model, test_loader)
    lw = torch.softmax(model.layer_w.detach().float().cpu(), 0)
    run = {'seed': seed, 'best_epoch': best_epoch, 'epochs_run': len(history), 'stop_reason': stop_reason,
           'initial_loss': init_loss, 'initial_loss_check': init_status,
           'best_val_f1_during_training': float(best_f1),
           'val': val_m, 'test': test_m,
           'layer_weights': lw.tolist(),
           'top5_layers': [[int(i), float(lw[i])] for i in torch.argsort(lw, descending=True)[:5]],
           'history': history, 'train_minutes': (time.time() - t_train) / 60,
           'peak_gpu_mem_gb': torch.cuda.max_memory_allocated() / 1e9 if AMP else None}
    print(f'  seed {seed} best epoch {best_epoch} -> val acc {val_m["acc"]:.4f} F1 {val_m["f1"]:.4f} | '
          f'test acc {test_m["acc"]:.4f} F1 {test_m["f1"]:.4f}   (v1 test: acc {V1_TEST["acc"]} F1 {V1_TEST["f1"]})', flush=True)
    print('  top-5 layers (softmax layer_w):', run['top5_layers'], flush=True)
    return run, best_state


def save_best(state, run):
    tmp = WORK / (BEST_CKPT.name + '.tmp')
    torch.save({'model_state_dict': state, 'seed': run['seed'], 'val_f1': run['val']['f1'],
                'test_acc': run['test']['acc'], 'test_f1': run['test']['f1'],
                'cluster_names': CLUSTER_NAMES, 'model_name': AUDIO_MODEL_NAME,
                'freeze_layers': FREEZE_LAYERS}, tmp)
    os.replace(tmp, BEST_CKPT)                  # atomic: only one checkpoint ever remains on disk


def write_results():
    tmp = WORK / (RESULTS_PATH.name + '.tmp')
    with open(tmp, 'w') as f:
        json.dump(RESULTS, f, indent=1, default=_json_default)
    os.replace(tmp, RESULTS_PATH)

## 7 · Run seeds 42, 43, 44
Results are written after **each** seed. `audio_v3_best.pt` is overwritten only when a seed's **validation**
macro-F1 beats every previous seed. Re-running this cell in the same session skips seeds already in the results file.

In [ ]:
RESULTS = (json.load(open(RESULTS_PATH)) if RESULTS_PATH.exists()
           else {'config': CONFIG, 'runs': [], 'errors': [], 'saved_checkpoint': None})
done = {r['seed'] for r in RESULTS['runs']}
best_val = RESULTS['saved_checkpoint']['val_f1'] if (RESULTS.get('saved_checkpoint') and BEST_CKPT.exists()) else -1.0

for seed in SEEDS:
    if seed in done:
        print(f'seed {seed}: already in {RESULTS_PATH.name} - skipped', flush=True)
        continue
    if hours() > TIME_BUDGET_H:
        print(f'seed {seed}: not started - time budget reached', flush=True)
        RESULTS['errors'].append({'seed': seed, 'error': 'not started: time budget'}); write_results()
        continue
    print(f'\n================ seed {seed} ================', flush=True)
    t_seed = time.time()
    try:
        run, state = run_seed(seed)
        run['seed_minutes'] = (time.time() - t_seed) / 60
        run['saved_when_finished'] = run['val']['f1'] > best_val
        if run['saved_when_finished']:
            save_best(state, run); best_val = run['val']['f1']
            RESULTS['saved_checkpoint'] = {'seed': seed, 'val_f1': run['val']['f1'], 'val_acc': run['val']['acc'],
                                           'test_acc': run['test']['acc'], 'test_f1': run['test']['f1'],
                                           'best_epoch': run['best_epoch'], 'path': str(BEST_CKPT)}
            print(f'  -> saved {BEST_CKPT.name} (best val macro-F1 so far {best_val:.4f})', flush=True)
        else:
            print(f'  -> checkpoint not replaced (val macro-F1 {run["val"]["f1"]:.4f} <= best so far {best_val:.4f})', flush=True)
        RESULTS['runs'].append(run)
        del state, run
    except Exception as e:
        traceback.print_exc()
        RESULTS['errors'].append({'seed': seed, 'error': f'{type(e).__name__}: {e}'})
    write_results()
    gc.collect()
    if AMP:
        torch.cuda.empty_cache()
        print(f'  GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated', flush=True)
    print(f'seed {seed} took {(time.time() - t_seed) / 60:.1f} min | total elapsed {hours():.2f} h', flush=True)

## 8 · SUMMARY (copy the whole block between the markers)

In [ ]:
runs = sorted(RESULTS['runs'], key=lambda r: r['seed'])


def mean_std(vals):
    a = np.array(vals, dtype=float)
    return {'mean': float(a.mean()), 'std': float(a.std(ddof=1)) if len(a) > 1 else 0.0, 'n': int(len(a))}


agg = {f'{s}_{m}': mean_std([r[s][m] for r in runs]) for s in ('val', 'test') for m in ('acc', 'f1')} if runs else {}
SUMMARY = {
    'model': f'AudioModelV3 ({AUDIO_MODEL_NAME}, freeze_layers={FREEZE_LAYERS}, weighted 25-layer sum, masked mean+std pooling)',
    'protocol': CONFIG['split'] + ' | ' + CONFIG['preprocessing'],
    'seeds_requested': SEEDS,
    'seeds_completed': [r['seed'] for r in runs],
    'errors': RESULTS['errors'],
    'per_seed': [{'seed': r['seed'], 'best_epoch': r['best_epoch'], 'epochs_run': r['epochs_run'],
                  'stop_reason': r['stop_reason'], 'initial_loss': r['initial_loss'],
                  'initial_loss_check': r['initial_loss_check'],
                  'val_acc': r['val']['acc'], 'val_f1': r['val']['f1'],
                  'test_acc': r['test']['acc'], 'test_f1': r['test']['f1'],
                  'top5_layers': r['top5_layers'], 'seed_minutes': r.get('seed_minutes')} for r in runs],
    'mean_std_over_seeds (sample std, ddof=1)': agg,
    'test_per_class_f1_mean_std': {c: mean_std([r['test']['report'][c]['f1-score'] for r in runs]) for c in CLUSTER_NAMES} if runs else {},
    'mean_layer_weights': np.mean([r['layer_weights'] for r in runs], 0).tolist() if runs else [],
    'v1_test': V1_TEST,
    'v3_mean_minus_v1_test': ({'acc': agg['test_acc']['mean'] - V1_TEST['acc'],
                               'f1': agg['test_f1']['mean'] - V1_TEST['f1']} if runs else None),
    'seeds_above_v1_test': ({'acc': sum(r['test']['acc'] > V1_TEST['acc'] for r in runs),
                             'f1': sum(r['test']['f1'] > V1_TEST['f1'] for r in runs)} if runs else None),
    'saved_checkpoint (selected by VALIDATION macro-F1)': RESULTS['saved_checkpoint'],
    'files_in_working': sorted(p.name for p in WORK.iterdir() if p.is_file()),
    'total_hours': hours(),
}

print('===== SUMMARY START =====')
print(f'Stage 2 v3 audio | {SUMMARY["model"]}')
print(f'{"seed":>4} | {"best ep":>7} | {"init loss":>9} | {"val acc":>7} | {"val F1":>7} | {"test acc":>8} | {"test F1":>7}')
for p in SUMMARY['per_seed']:
    print(f'{p["seed"]:>4} | {p["best_epoch"]:>7} | {p["initial_loss"]:>7.4f} {p["initial_loss_check"][0]} | '
          f'{p["val_acc"]:>7.4f} | {p["val_f1"]:>7.4f} | {p["test_acc"]:>8.4f} | {p["test_f1"]:>7.4f}')
if runs:
    print(f'mean +/- std (n={len(runs)}): val acc {agg["val_acc"]["mean"]:.4f} +/- {agg["val_acc"]["std"]:.4f} | '
          f'val F1 {agg["val_f1"]["mean"]:.4f} +/- {agg["val_f1"]["std"]:.4f} | '
          f'test acc {agg["test_acc"]["mean"]:.4f} +/- {agg["test_acc"]["std"]:.4f} | '
          f'test F1 {agg["test_f1"]["mean"]:.4f} +/- {agg["test_f1"]["std"]:.4f}')
    print(f'v1 test: acc {V1_TEST["acc"]:.4f} | F1 {V1_TEST["f1"]:.4f}  ->  v3 mean - v1: '
          f'acc {SUMMARY["v3_mean_minus_v1_test"]["acc"]:+.4f} | F1 {SUMMARY["v3_mean_minus_v1_test"]["f1"]:+.4f}')
sc = RESULTS['saved_checkpoint']
print('saved checkpoint:', f'seed {sc["seed"]} (val F1 {sc["val_f1"]:.4f}, test acc {sc["test_acc"]:.4f}, '
      f'test F1 {sc["test_f1"]:.4f}) -> {BEST_CKPT}' if sc else 'NONE')
if RESULTS['errors']:
    print('ERRORS:', RESULTS['errors'])
print(json.dumps({'summary': SUMMARY, 'all_results': RESULTS}, indent=1, default=_json_default))
print('===== SUMMARY END =====')